# 3 — Modelling (Training & Evaluation)

Tahap modelling: split train/test, pelatihan model, dan evaluasi metrik.

**Input:** `2_data_preprocessing/output/2.2_final_feature_set.csv`

**Output:**
- `3_modelling/output/model_metrics.csv`
- `3_modelling/output/3_model_predictions.csv`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Coba gunakan XGBoost; jika tidak tersedia, fallback ke RandomForest agar notebook tetap runnable.
try:
    import xgboost as xgb
    use_xgboost = True
    model_name = 'XGBoost'
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    use_xgboost = False
    model_name = 'RandomForest'

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '2_data_preprocessing' / 'output' / '2.2_final_feature_set.csv').exists():
            return p
    raise FileNotFoundError('Could not find 2.2_final_feature_set.csv in current or parent directories.')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / '2_data_preprocessing' / 'output' / '2.2_final_feature_set.csv'
output_dir = ROOT / '3_modelling' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
metrics_path = output_dir / 'model_metrics.csv'
pred_path = output_dir / '3_model_predictions.csv'

df = pd.read_csv(input_path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
df = df.sort_values(by=['provinsi_id', 'tanggal']).reset_index(drop=True)

print(f'Loaded: {input_path}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} cols')

In [ ]:
# Siapkan data modelling: minimal harus punya target dan fitur lag
lag_columns = [col for col in df.columns if '_lag_' in col]
required_cols = lag_columns + ['twp90_pct']
df_model = df.dropna(subset=required_cols).copy()

# Pastikan tahun numeric
df_model['tahun'] = pd.to_numeric(df_model['tahun'], errors='coerce')
df_model = df_model.dropna(subset=['tahun'])
df_model['tahun'] = df_model['tahun'].astype(int)

kolom_non_prediktor = ['provinsi_id', 'nama_provinsi', 'tanggal', 'tahun', 'bulan', 'twp90_pct']
X_cols = [col for col in df_model.columns if col not in kolom_non_prediktor]

print(f'Total rows available for modelling: {len(df_model):,}')
print(f'Total features used: {len(X_cols)}')

# Split temporal: Train 2022-2024 | Test 2025
train_data = df_model[df_model['tahun'] <= 2024].copy()
test_data = df_model[df_model['tahun'] == 2025].copy()

if train_data.empty or test_data.empty:
    print('Warning: strict temporal split produced an empty set. Using 80/20 chronological fallback split.')
    df_model = df_model.sort_values('tanggal').reset_index(drop=True)
    split_idx = int(0.8 * len(df_model))
    train_data = df_model.iloc[:split_idx].copy()
    test_data = df_model.iloc[split_idx:].copy()

X_train = train_data[X_cols]
y_train = train_data['twp90_pct']
X_test = test_data[X_cols]
y_test = test_data['twp90_pct']

print(f'Dimensi X_train: {X_train.shape}')
print(f'Dimensi X_test: {X_test.shape}')

assert not X_train.empty and not X_test.empty, 'Dataset train/test kosong setelah preprocessing.'
assert (df_model['provinsi_id'] == 19).sum() == 0, 'provinsi_id 19 still present (expected removed upstream).'

In [ ]:
# Latih model baseline
if use_xgboost:
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
    )
else:
    model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2 = float(r2_score(y_test, y_pred))

print(f'--- Evaluasi Baseline Model ({model_name}) ---')
print(f'RMSE: {rmse:.4f}')
print(f'MAE: {mae:.4f}')
print(f'R2: {r2:.4f}')

metrics_df = pd.DataFrame([{
    'model': model_name,
    'rmse': rmse,
    'mae': mae,
    'r2': r2,
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'n_features': int(len(X_cols)),
}])
metrics_df.to_csv(metrics_path, index=False)
print(f'Saved: {metrics_path}')

out_cols = [c for c in ['provinsi_id', 'nama_provinsi', 'tanggal', 'tahun', 'bulan', 'twp90_pct'] if c in test_data.columns]
pred_df = test_data[out_cols].copy()
pred_df = pred_df.rename(columns={'twp90_pct': 'y_true'})
pred_df['y_pred'] = y_pred
pred_df['error'] = pred_df['y_pred'] - pred_df['y_true']
pred_df.to_csv(pred_path, index=False)
print(f'Saved: {pred_path}')